# Clusters de perfis de emissão em C

Buscamos grupos interpretáveis, sem pressupor que exista uma divisão natural dos 67 setores. A análise usa as dimensões originais, não a projeção 3D.

**Quatro cenários:** linhas e colunas, com C completa (referência) e com diagonal zerada (sensibilidade). Linhas comparam os destinos da demanda final; colunas comparam as origens emissoras. Remover a diagonal muda a pergunta para as relações com outras atividades e insere um zero em uma posição diferente para cada perfil; não é uma correção automática da matriz. Perfis que se tornam nulos são excluídos do cosseno e identificados.

C herda as hipóteses do notebook da MIP: intensidades interpoladas, aproximação monetária e exterior representativo com tecnologia brasileira. Clusters descrevem este modelo, não relações causais nem observações independentes de emissões.

Execute a partir da raiz ou desta pasta, após a exportação de C pelo notebook principal. Não há alteração das entradas nem da análise de similaridade anterior.

In [1]:
from pathlib import Path
from hashlib import sha256
import platform
from html import escape
import numpy as np
import pandas as pd
import scipy
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, cut_tree, fcluster
from IPython.display import display

raiz = Path.cwd().resolve()
if not (raiz / "outputs" / "matriz_emissoes_consumo_2015.csv").exists():
    raiz = raiz.parent
pasta = raiz / "similaridade_consumo"
entrada = raiz / "outputs" / "matriz_emissoes_consumo_2015.csv"
C = pd.read_csv(entrada, dtype={"atividade_emissora": str}, index_col="atividade_emissora")
nomes = pd.read_csv(raiz / "outputs" / "setores_2015.csv", dtype={"atividade": str}).set_index("atividade")["descricao"]
assert C.shape == (67, 67) and C.index.equals(C.columns)
assert C.index.is_unique and C.index.isin(nomes.index).all()
assert np.isfinite(C.to_numpy()).all() and (C.to_numpy() >= 0).all()
C_original = C.copy()
diagonal = np.diag(C.to_numpy())
fracao_diagonal = diagonal.sum() / C.to_numpy().sum()
print("SHA-256:", sha256(entrada.read_bytes()).hexdigest())
print("Python:", platform.python_version(), "NumPy:", np.__version__, "pandas:", pd.__version__, "SciPy:", scipy.__version__)
print(f"Diagonal / total de C: {fracao_diagonal:.2%}")


SHA-256: cbc6a595f72f234caf1bfa4ec84885d5e804540bbfc9ccd4c480273298fa0238
Python: 3.12.10 NumPy: 2.2.6 pandas: 3.0.0 SciPy: 1.17.1
Diagonal / total de C: 52.34%


## 1. Existe uma partição global convincente?

Usamos dissimilaridade $d=1-\cos$, ligação média e cortes de **2 a 12 grupos**. A faixa é exploratória, não uma busca exaustiva. Registramos todos os resultados, incluindo o tamanho do maior grupo e quantos grupos são unitários; o maior valor de silhueta na faixa não prova que esse número de grupos seja adequado.

A silhueta de cada perfil é $(b-a)/\max(a,b)$: $a$ é a distância média aos demais membros do grupo e $b$, a menor distância média a outro grupo. Grupos unitários recebem zero. Valores próximos de zero indicam pouca separação; negativos indicam maior proximidade média com outro grupo.

Referências de implementação: [ligação média e completa no SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html) e [definição de silhueta](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_samples.html). O cálculo abaixo usa apenas NumPy; scikit-learn não é dependência do notebook.

In [2]:
def silhuetas(distancias, grupos):
    """Silhueta por perfil a partir das dissimilaridades entre todos os pares."""
    resultado = np.zeros(len(grupos))
    for i in range(len(grupos)):
        mesmo_grupo = grupos == grupos[i]
        mesmo_grupo[i] = False
        if not mesmo_grupo.any():
            continue
        a = distancias[i, mesmo_grupo].mean()
        b = min(distancias[i, grupos == grupo].mean()
                for grupo in np.unique(grupos) if grupo != grupos[i])
        resultado[i] = (b-a)/max(a, b) if max(a, b) > 0 else 0
    return resultado

cenarios = {}
qualidade = []
for recorte in ["completa", "sem_diagonal"]:
    valores = C.to_numpy(copy=True)
    if recorte == "sem_diagonal":
        np.fill_diagonal(valores, 0)
    for eixo, perfis in [("linhas", valores), ("colunas", valores.T)]:
        chave = f"{eixo}_{recorte}"
        validos = np.linalg.norm(perfis, axis=1) > 0
        codigos = C.index[validos]
        X = perfis[validos]
        d = np.clip(pdist(X, metric="cosine"), 0, 1)
        D = squareform(d)
        arvore_media = linkage(d, method="average")
        cenarios[chave] = dict(eixo=eixo, recorte=recorte, X=X, codigos=codigos,
                               excluidos=C.index[~validos].tolist(), D=D,
                               arvore_completa=linkage(d, method="complete"))
        for k in range(2, min(12, len(X)-1)+1):
            grupos = cut_tree(arvore_media, n_clusters=k).ravel()
            tamanhos = np.bincount(grupos)
            qualidade.append(dict(cenario=chave, k=k, silhueta= silhuetas(D, grupos).mean(),
                                   maior_grupo=tamanhos.max(), unitarios=(tamanhos == 1).sum(),
                                   tamanhos=", ".join(map(str, sorted(tamanhos, reverse=True)))))
qualidade = pd.DataFrame(qualidade)
melhores = qualidade.loc[qualidade.groupby("cenario")["silhueta"].idxmax()].reset_index(drop=True)
display(melhores)
for chave, dados in cenarios.items():
    print(chave, "— perfis nulos excluídos:", dados["excluidos"])


                cenario   k  ...  unitarios                             tamanhos
0      colunas_completa   2  ...          1                                66, 1
1  colunas_sem_diagonal   2  ...          1                                65, 1
2       linhas_completa  12  ...          8  52, 3, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1
3   linhas_sem_diagonal   4  ...          1                          58, 4, 3, 1

[4 rows x 6 columns]
linhas_completa — perfis nulos excluídos: []
colunas_completa — perfis nulos excluídos: []
linhas_sem_diagonal — perfis nulos excluídos: ['9700']
colunas_sem_diagonal — perfis nulos excluídos: ['9700']


## 2. Pequenos grupos de alta similaridade

Em vez de forçar todos os setores a uma divisão global, usamos **ligação completa** com corte em $d=0,20$: todos os pares de um grupo devem ter cosseno ≥ **0,80**. Grupos unitários são registrados como “sem grupo neste limiar”; não significa ausência de relações econômicas. Esta é uma partição hierárquica exploratória, não a enumeração de todos os subconjuntos possíveis de alta similaridade. Empates e o método de ligação podem afetar membros de grupos próximos.

Mostramos também cortes em **0,70 e 0,90**. Para o corte principal, realizamos **100 perturbações**, com semente 42: cada dimensão recebe um peso uniforme entre 0,9 e 1,1, comum a todos os perfis da repetição. Multiplicamos as coordenadas pela raiz do peso e recalculamos o cosseno e os grupos. A retenção é a frequência média com que os pares internos continuam juntos; não exige que o grupo inteiro permaneça idêntico. Isso mede sensibilidade local à ponderação das dimensões, **não confiança estatística**, erro amostral nem robustez dos coeficientes de emissão.

Para interpretar um grupo, calculamos a média dos perfis convertidos em participações (cada perfil soma 1). Todos os membros têm o mesmo peso: as participações mostradas não são a distribuição das emissões totais agregadas do grupo. Nas linhas, as dimensões são destinos; nas colunas, origens emissoras.

In [3]:
limiar_principal = 0.80
repeticoes = 100
pesos_dimensoes = np.random.default_rng(42).uniform(0.9, 1.1, size=(repeticoes, len(C)))
resumos, membros, sensibilidade, perfis_grupos = [], [], [], []

for chave, dados in cenarios.items():
    X, codigos, D = dados["X"], dados["codigos"], dados["D"]
    for limiar in [0.70, 0.80, 0.90]:
        rotulos = fcluster(dados["arvore_completa"], 1-limiar, criterion="distance")
        tamanhos = np.unique(rotulos, return_counts=True)[1]
        sensibilidade.append(dict(cenario=chave, limiar=limiar, grupos_nao_unitarios=(tamanhos > 1).sum(),
                                   setores_agrupados=tamanhos[tamanhos > 1].sum(),
                                   sem_grupo=(tamanhos == 1).sum()))
    rotulos = fcluster(dados["arvore_completa"], 1-limiar_principal, criterion="distance")
    frequencia = np.zeros(D.shape)
    for pesos in pesos_dimensoes:
        distancias = np.clip(pdist(X*np.sqrt(pesos), metric="cosine"), 0, 1)
        arvore = linkage(distancias, method="complete")
        pert = fcluster(arvore, 1-limiar_principal, criterion="distance")
        frequencia += pert[:, None] == pert[None, :]
    frequencia /= repeticoes

    # IDs em ordem do código do primeiro membro, sem significado econômico.
    grupos = sorted(np.unique(rotulos), key=lambda g: min(codigos[rotulos == g]))
    numero = 0
    for grupo in grupos:
        indices = np.flatnonzero(rotulos == grupo)
        if len(indices) < 2:
            i = indices[0]
            membros.append(dict(cenario=chave, grupo="sem_grupo", atividade=codigos[i], descricao=nomes[codigos[i]]))
            continue
        numero += 1
        id_grupo = f"G{numero:02d}"
        triangular = np.triu_indices(len(indices), k=1)
        cos = (1-D[np.ix_(indices, indices)])[triangular]
        assert cos.min() >= limiar_principal - 1e-12
        retencao = frequencia[np.ix_(indices, indices)][triangular].mean()
        proporcoes = X[indices] / X[indices].sum(axis=1)[:, None]
        perfil_medio = proporcoes.mean(axis=0)
        np.testing.assert_allclose(perfil_medio.sum(), 1)
        principais = np.argsort(-perfil_medio, kind="stable")[:3]
        resumo = dict(cenario=chave, grupo=id_grupo, n=len(indices),
                      membros="; ".join(f"{codigos[i]} — {nomes[codigos[i]]}" for i in indices),
                      cosseno_min=cos.min(), cosseno_medio=cos.mean(), retencao_pares=retencao,
                      principais_componentes="; ".join(f"{C.columns[j]} — {nomes[C.columns[j]]}: {perfil_medio[j]:.1%}" for j in principais))
        resumos.append(resumo)
        for i in indices:
            membros.append(dict(cenario=chave, grupo=id_grupo, atividade=codigos[i], descricao=nomes[codigos[i]]))
        for j, codigo in enumerate(C.columns):
            perfis_grupos.append(dict(cenario=chave, grupo=id_grupo, componente=codigo,
                                       descricao=nomes[codigo], participacao_media=perfil_medio[j]))
    for codigo in dados["excluidos"]:
        membros.append(dict(cenario=chave, grupo="perfil_nulo", atividade=codigo, descricao=nomes[codigo]))

resumos = pd.DataFrame(resumos)
membros = pd.DataFrame(membros)
sensibilidade = pd.DataFrame(sensibilidade)
perfis_grupos = pd.DataFrame(perfis_grupos)
assert membros.groupby("cenario").size().eq(len(C)).all()
assert not membros.duplicated(["cenario", "atividade"]).any()
pd.testing.assert_frame_equal(C, C_original)
display(sensibilidade)
display(resumos[resumos.cenario == "linhas_completa"])


                 cenario  limiar  ...  setores_agrupados  sem_grupo
0        linhas_completa     0.7  ...                 26         41
1        linhas_completa     0.8  ...                 19         48
2        linhas_completa     0.9  ...                  9         58
3       colunas_completa     0.7  ...                  6         61
4       colunas_completa     0.8  ...                  4         63
5       colunas_completa     0.9  ...                  0         67
6    linhas_sem_diagonal     0.7  ...                 61          5
7    linhas_sem_diagonal     0.8  ...                 59          7
8    linhas_sem_diagonal     0.9  ...                 42         24
9   colunas_sem_diagonal     0.7  ...                 57          9
10  colunas_sem_diagonal     0.8  ...                 48         18
11  colunas_sem_diagonal     0.9  ...                 32         34

[12 rows x 5 columns]
           cenario  ...                             principais_componentes
0  linhas_completa

## 3. Leitura dos resultados e relatório

O relatório apresenta primeiro C completa e seus grupos locais, depois os cenários sem diagonal como sensibilidade. Grupos com nomes setoriais plausíveis devem ser justificados pelas dimensões compartilhadas, não apenas pelo rótulo. Uma silhueta maior acompanhada de um único grupo gigante e alguns isolados não demonstra uma boa segmentação econômica.

A retenção é calculada dentro de cada cenário. Não mede persistência entre C completa e C sem diagonal; essas associações podem ser conferidas na tabela de membros por setor. A análise não ajusta transformações para obter um resultado desejado.

In [4]:
for nome, tabela in [("qualidade_clusters_C", qualidade), ("grupos_clusters_C", resumos),
                     ("membros_clusters_C", membros), ("sensibilidade_clusters_C", sensibilidade),
                     ("perfis_clusters_C", perfis_grupos)]:
    tabela.to_csv(pasta / f"{nome}.csv", index=False, encoding="utf-8")

partes = ["<!doctype html><html lang='pt-BR'><meta charset='utf-8'><title>Clusters de C</title>",
          "<style>body{font-family:Arial,sans-serif;max-width:1200px;margin:32px auto;padding:0 20px;color:#172033;line-height:1.5}"
          "table{border-collapse:collapse;font-size:13px;width:100%;margin:16px 0}td,th{border:1px solid #ddd;padding:8px;text-align:left}"
          "article{border:1px solid #ccd6e0;border-radius:8px;padding:16px;margin:18px 0;background:#f8fafc}"
          "summary{cursor:pointer;font-weight:bold}.nota{padding:16px;background:#fff2cc}small{overflow-wrap:anywhere}</style>",
          "<h1>Há grupos de setores com perfis semelhantes?</h1>",
          f"<p class='nota'>A diagonal representa <strong>{fracao_diagonal:.1%}</strong> do total de C. "
          "As partições globais examinadas devem ser lidas junto com seus tamanhos: não pressupomos uma divisão natural da economia. "
          "Os pequenos grupos abaixo exigem cosseno ≥ 0,80 entre todos os pares.</p>",
          f"<small>Entrada: {escape(str(entrada.relative_to(raiz)))}<br>SHA-256: {sha256(entrada.read_bytes()).hexdigest()}</small>",
          "<h2>1. Diagnóstico global: ligação média, 2 a 12 grupos</h2>",
          "<p>Melhor silhueta dentro da faixa testada, sem impor tamanhos mínimos. Grupos unitários recebem silhueta zero. "
          "Um grupo enorme acompanhado de isolados não é uma segmentação equilibrada, mesmo com silhueta positiva.</p>",
          melhores.to_html(index=False, float_format=lambda x: f"{x:.3f}"),
          "<details><summary>Todos os cortes avaliados</summary>",
          qualidade.to_html(index=False, float_format=lambda x: f"{x:.3f}"), "</details>",
          "<h2>2. Grupos locais e sua interpretação</h2>",
          "<p>Ligação completa, cosseno mínimo 0,80. Retenção: frequência média de permanência dos pares internos "
          "em 100 reponderações de ±10% das dimensões, semente 42. É sensibilidade local, não confiança estatística. "
          "As participações são médias dos perfis de cada membro normalizados para somar 1, com pesos iguais.</p>"]
# Exemplos locais destacados; os demais grupos e os cenários alternativos aparecem abaixo.
exemplos = [("Construção e minerais", {"0580", "2300", "4180"}),
            ("Automóveis e autopeças", {"2991", "2992"}),
            ("Petróleo e refino", {"0680", "1991"})]
leitura = ["<h2>Achados para começar a leitura</h2><p>As linhas fornecem exemplos locais interpretáveis. "
           "Eles não validam uma partição global nem demonstram fluxos diretos entre os membros. "
           "Nos percentuais abaixo, cada perfil tem o mesmo peso.</p>"]
for titulo, codigos_exemplo in exemplos:
    associados = membros[(membros.cenario == "linhas_completa") & membros.atividade.isin(codigos_exemplo)]
    if associados.grupo.nunique() == 1 and associados.iloc[0].grupo != "sem_grupo":
        grupo = resumos[(resumos.cenario == "linhas_completa") & (resumos.grupo == associados.iloc[0].grupo)].iloc[0]
        leitura.append(f"<p><strong>{titulo}:</strong> cosseno mínimo {grupo.cosseno_min:.3f}; "
                       f"retenção dos pares {grupo.retencao_pares:.1%}. "
                       f"{escape(grupo.principais_componentes)}</p>")
leitura.append("<p>Esses grupos usam C completa. Remover a diagonal altera seus perfis e pode desfazer "
               "as associações: a estabilidade acima vale apenas para as pequenas reponderações dentro desse cenário.</p>")
partes[5:5] = leitura
for chave in ["linhas_completa", "colunas_completa", "linhas_sem_diagonal", "colunas_sem_diagonal"]:
    dados = cenarios[chave]
    componentes = "destinos da demanda final" if dados["eixo"] == "linhas" else "origens emissoras"
    partes += [f"<h3>{escape(chave.replace('_', ' '))}</h3>", f"<p>Comparação por {componentes}. "
               + ("C completa: cenário de referência." if dados["recorte"] == "completa" else "Sensibilidade: exclui a atribuição à própria atividade; muda a pergunta e pode mudar os grupos.") + "</p>"]
    for _, grupo in resumos[resumos.cenario == chave].iterrows():
        lista = membros[(membros.cenario == chave) & (membros.grupo == grupo.grupo)]
        partes += [f"<article><strong>{grupo.grupo} · {grupo.n} setores</strong><ul>"]
        partes += [f"<li>{escape(l.atividade)} — {escape(l.descricao)}</li>" for l in lista.itertuples()]
        partes += [f"</ul><p>Cosseno mínimo: <strong>{grupo.cosseno_min:.3f}</strong> · Média: {grupo.cosseno_medio:.3f} · "
                   f"Retenção dos pares: <strong>{grupo.retencao_pares:.1%}</strong></p>",
                   f"<p><strong>Principais {componentes}:</strong><br>{escape(grupo.principais_componentes)}</p></article>"]
    restantes = membros[(membros.cenario == chave) & membros.grupo.isin(["sem_grupo", "perfil_nulo"])]
    partes += [f"<details><summary>Setores sem grupo neste limiar ou com perfil nulo ({len(restantes)})</summary>",
               restantes[["atividade", "descricao", "grupo"]].to_html(index=False), "</details>"]
partes += ["<h2>3. Sensibilidade ao limiar de similaridade</h2>", sensibilidade.to_html(index=False),
           "<h2>Limites e reprodução</h2><p>Grupos são exploratórios e dependem de recorte, ligação e limiar. "
           "Não representam comunidades de fluxos nem evidência causal. C usa intensidades interpoladas, aproximação monetária "
           "e exterior representativo do modelo original. Execute analise_clusters_C.ipynb para reproduzir tabelas e relatório.</p>",
           "<p><a href='https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html'>Métodos de ligação</a> · "
           "<a href='https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_samples.html'>Silhueta</a></p></html>"]
(pasta / "clusters_C.html").write_text("\n".join(partes), encoding="utf-8")
print("Relatório: clusters_C.html")


Relatório: clusters_C.html
